In [2]:
import numpy as np
import gymnasium as gym

In [3]:
class FourierBasis:
    def __init__(self, state_dim, order):
        self.order = order
        self.state_dim = state_dim
        self.num_features = (order + 1) ** state_dim
        self.c = np.array(np.meshgrid(*[range(order + 1)] * state_dim)).T.reshape(-1, state_dim)

    def get_features(self, state):
        return np.cos(np.pi * np.dot(self.c, state))  # shape: (num_features,)

In [4]:
class QLearningAgent:
    def __init__(self, env, order=3, alpha=0.01, gamma=0.99, epsilon=0.1):
        self.env = env
        self.state_dim = env.observation_space.shape[0]
        self.num_actions = env.action_space.n
        self.basis = FourierBasis(self.state_dim, order)
        self.num_features = self.basis.num_features
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.weights = np.zeros((self.num_actions, self.num_features))

    def normalize_state(self, state):
        low = self.env.observation_space.low
        high = self.env.observation_space.high
        return (state - low) / (high - low)

    def get_q_values(self, state):
        norm_state = self.normalize_state(state)
        features = self.basis.get_features(norm_state)
        q_values = np.dot(self.weights, features)  # shape: (num_actions,)
        return q_values, features

    def choose_action(self, state):
        q_values, _ = self.get_q_values(state)
        if np.random.rand() < self.epsilon:
            return self.env.action_space.sample()
        else:
            return np.argmax(q_values)

    def train(self, episodes=500):
        for episode in range(episodes):
            state = self.env.reset()[0]
            done = False
            rewards = 0
            while not done:
                action = self.choose_action(state)
                next_state, reward, done, _, info = self.env.step(action)
                rewards += reward
                q_values, features = self.get_q_values(state)
                next_q_values, _ = self.get_q_values(next_state)

                td_target = reward + self.gamma * np.max(next_q_values) * (not done)
                td_error = td_target - q_values[action]

                self.weights[action] += self.alpha * td_error * features
                state = next_state

            if (episode + 1) % 50 == 0:
                print(f"Episode {episode + 1} complete with {rewards} as rewards!")


In [6]:
env = gym.make("MountainCar-v0")
agent = QLearningAgent(env, order=3, alpha=0.01, gamma=0.99, epsilon=0.1)
agent.train(episodes=500)

Episode 50 complete with -185.0 as rewards!
Episode 100 complete with -135.0 as rewards!
Episode 150 complete with -138.0 as rewards!
Episode 200 complete with -238.0 as rewards!
Episode 250 complete with -140.0 as rewards!
Episode 300 complete with -142.0 as rewards!
Episode 350 complete with -166.0 as rewards!
Episode 400 complete with -112.0 as rewards!
Episode 450 complete with -145.0 as rewards!
Episode 500 complete with -150.0 as rewards!
